### Running TissueAgent on the Cell Annotation Baseline

In [ ]:
TASK = "cell_annotation"


In [2]:
import os
import shutil
import sys
from functools import partial
from pathlib import Path
from queue import Queue

from langchain_core.messages import HumanMessage
from openai import RateLimitError

sys.path.append(str(Path.cwd().parent / "src"))

### configure logging
from config import DATA_DIR, RECURSION_LIMIT
from notebook_utils import tee_output
log_path = Path.cwd() / f"outputs/{TASK}/transcript.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
log_path.write_text("")
tee = partial(tee_output, path=log_path)

### reset data directory
from notebook_utils import _reset_data_directories
_reset_data_directories()

import os

In [3]:
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    entered_key = getpass("OpenAI API key (input hidden): ").strip()
    if entered_key:
        os.environ["OPENAI_API_KEY"] = entered_key

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not set in this Jupyter kernel. "
        "Export it before launching Jupyter, restart the kernel, or enter it when prompted."
    )

print("OPENAI_API_KEY is set for this kernel.")


OPENAI_API_KEY is set for this kernel.


### Create minimal AnnData object with all annotations and intermediate preprocessing data removed

In [4]:
import anndata as ad

def strip_annotations_for_unannotated_spatial(adata: ad.AnnData) -> ad.AnnData:
    # Keep only metadata that is reasonable for an unannotated spatial dataset.
    obs_keep = [
        c for c in [
            "sample_id",
            "batch",
            "n_counts",
        ]
        if c in adata.obs.columns
    ]

    obsm_keep = {
        k: adata.obsm[k].copy()
        for k in ["spatial"]
        if k in adata.obsm
    }


    stripped = ad.AnnData(
        X=adata.X.copy(),
        obs=adata.obs[obs_keep].copy(),   # keeps obs_names
        var=adata.var.copy(),   # keeps var_names / gene symbols
        obsm=obsm_keep,
    )

    return stripped

spatial_data_file_path = Path.cwd() / "data" / "mouse_cns_raw.h5ad"
annotated_spatial_dataset = ad.read_h5ad(spatial_data_file_path)

unannotaed_spatial_dataset = strip_annotations_for_unannotated_spatial(annotated_spatial_dataset)

write_file_path = DATA_DIR / "dataset" / f"min_{spatial_data_file_path.name}"
unannotaed_spatial_dataset.write_h5ad(write_file_path)

In [5]:
### Task Files
data_src_dir = Path.cwd() / "data"
print("Files present in DATA_DIR:", end="\n - ")
files = [str(f.relative_to(DATA_DIR)) for f in DATA_DIR.rglob("*") if f.is_file()]
print("\n - ".join(files))

Files present in DATA_DIR:
 - heart_dev_sc_reference.h5ad
 - dataset/min_mouse_cns_raw.h5ad


In [6]:
# ### Prompt
# prompt = f"""
# Please annotate the cells in the spatial transcriptomics dataset located in the min_{spatial_data_file_path.name} afile. The dataset is derived from the developing human heart.
# """.replace("\n", " ")

### Prompt
prompt = f"""
Please annotate the cells in the spatial transcriptomics dataset located in the min_{spatial_data_file_path.name} anndata file. The dataset is derived from the mouse central nervous system.
""".replace("\n", " ")

### Initialize TissueAgent

In [7]:
from graph.graph import create_tissueagent_graph

def _bind_retry(model):
    """Use provider retry defaults; avoid repeating insufficient_quota failures."""
    return model

state_queue = Queue()

with tee():
    graph = create_tissueagent_graph(state_queue, _bind_retry)
    tissueagent = graph.compile()


/home/etrop/TissueAgent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/etrop/TissueAgent/.venv/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/home/etrop/TissueAgent/.venv/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/home/etrop/TissueAgent/.venv/lib/python3.12/site-packages/anndata/__init__.py:44: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  return module_get_attr_redi

In [ ]:
def _is_insufficient_quota(exc: RateLimitError) -> bool:
    body = getattr(exc, "body", None)
    if isinstance(body, dict) and body.get("code") == "insufficient_quota":
        return True
    return "insufficient_quota" in str(exc)

try:
    with tee():
        result = tissueagent.invoke(
            {"messages": [("user", prompt)]},
            config={"recursion_limit": RECURSION_LIMIT},
        )
except RateLimitError as exc:
    if _is_insufficient_quota(exc):
        raise RuntimeError(
            "OpenAI returned insufficient_quota. The API key is being read, but the "
            "OpenAI project/account has no available quota for this model. Check the "
            "project selected by OPENAI_PROJECT_ID, billing, usage limits, or switch "
            "DefaultModelCtor in src/config.py to a model your project can use."
        ) from exc
    raise

print(result["messages"][-1].content)


2026-07-08 12:13:08 | INFO     | HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"

2026-07-08 12:13:08 | INFO     | Message Info
Type: ai

Name: planner_agent

ID:   run--e882ca5a-3ade-4976-aa00-af82af179cd7-0

Content:
ToolCalls:
  1. {'name': 'template_selector_tool', 'args': {'query_text': 'Annotate the cells in the spatial transcriptomics dataset min_mouse_cns_raw.h5ad from mouse CNS using reference-based label transfer, producing an annotated AnnData and brief summary.', 'inputs_available': ['AnnData (.h5ad) spatial dataset', 'species: mouse', 'tissue: CNS']}, 'id': 'call_fJ5obpMcGLmjZ72qxF2iOo2X', 'type': 'tool_call'}

2026-07-08 12:13:08 | INFO     | Message Info
Type: tool

Name: template_selector_tool

ID:   None

Content:
{
  "decision": "ADAPT",
  "template_id": "CELL_ANNOTATION",
  "score": 0.19,
  "scores": {
    "tags": 0.174,
    "keywords": 0.15,
    "io": 0.0,
    "recency": 0.604
  },
  "why": "Partial match (score=0.19); adapt parameter

In [2]:
import anndata as ad



ann_ad = ad.read_h5ad('./outputs/cell_annotation/mouse_cns/annotated_object.h5ad')
raw_ad = ad.read_h5ad('./data/mouse_cns_raw.h5ad')

ann_ad,raw_ad

(AnnData object with n_obs × n_vars = 781428 × 995
     obs: 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'n_genes', 'batch', 'dataset', 'harmony_predicted_cell_type', 'harmony_prediction_confidence', 'label'
     var: 'original_var_name', 'gene_identifier_source', 'gene_identifier_source_name', 'gene_identifier_pre_case_alignment', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
     uns: 'hvg', 'log1p'
     obsm: 'spatial',
 AnnData object with n_obs × n_vars = 1091280 × 1022
     obs: 'sample', 'slide', 'expression_source', 'biosample_id', 'donor_id', 'species', 'species__ontology_label', 'disease', 'disease__ontology_label', 'organ', 'organ__ontology_label', 'library_preparation_protocol', 'library_preparatio

### Evaluating Cell Annotation Baselines

In [17]:
import json
import scanpy as sc
import scanpy._compat as _scanpy_compat
from numba import njit as _njit

# omicverse expects scanpy._compat.njit, which is absent in scanpy 1.10.3.
# Patch it at runtime before omicverse is imported later in the notebook.

if not hasattr(_scanpy_compat, "njit"):
    _scanpy_compat.njit = _njit

In [18]:
# read in spatial transcriptomics eval dataset
data = sc.read_h5ad('../data/dataset/overall_merfish.h5ad')

# load cell type label mappings between set of cell lines predicted by baselines and set of cell types found in evaluation dataset
with open('../data/tables/harmonized_mapping.json','rb') as f:
    mapping = json.load(f)

#### 1. CellTypist


In [9]:
import celltypist
from celltypist import models

In [10]:
celltypist_adata = data.copy()
sc.pp.normalize_total(celltypist_adata,target_sum=10000)
sc.pp.log1p(celltypist_adata)

In [11]:
models.download_models(force_update = True)
models.Model.load(model = 'Healthy_Adult_Heart.pkl')
predictions = celltypist.annotate(celltypist_adata, model = 'Healthy_Adult_Heart.pkl', majority_voting = True)

📜 Retrieving model list from server https://celltypist.cog.sanger.ac.uk/models/models.json
📚 Total models in list: 61
📂 Storing models in /home/etrop/.celltypist/data/models
💾 Downloading model [1/61]: Immune_All_Low.pkl
💾 Downloading model [2/61]: Immune_All_High.pkl
💾 Downloading model [3/61]: Adult_COVID19_PBMC.pkl
💾 Downloading model [4/61]: Adult_CynomolgusMacaque_Hippocampus.pkl
💾 Downloading model [5/61]: Adult_Human_MTG.pkl
💾 Downloading model [6/61]: Adult_Human_PancreaticIslet.pkl
💾 Downloading model [7/61]: Adult_Human_PrefrontalCortex.pkl
💾 Downloading model [8/61]: Adult_Human_Skin.pkl
💾 Downloading model [9/61]: Adult_Human_Vascular.pkl
💾 Downloading model [10/61]: Adult_Mouse_Gut.pkl
💾 Downloading model [11/61]: Adult_Mouse_OlfactoryBulb.pkl
💾 Downloading model [12/61]: Adult_Pig_Hippocampus.pkl
💾 Downloading model [13/61]: Adult_RhesusMacaque_Hippocampus.pkl
💾 Downloading model [14/61]: Adult_cHSPCs_Illumina.pkl
💾 Downloading model [15/61]: Adult_cHSPCs_Ultima.pkl
💾 Dow

In [41]:
celltypist_preds = predictions.predicted_labels.predicted_labels.map(lambda x: mapping['mapping_celltypist'][x]) # map predicted cell types to set of cell types in eval dataset

data.obs['celltypist_pred_harmonized'] = celltypist_preds

#### 2. GTPCellType

*** You must export your OPENAI_API_KEY in your environment before running the GPTCellType code blocks `export OPENAI_API_KEY='<your API token>'` ***

In [13]:
import omicverse as ov
import os

In [ ]:
from getpass import getpass

# GPTCellType uses the same OPENAI_API_KEY configured above.
if not os.environ.get("OPENAI_API_KEY"):
    entered_key = getpass("OpenAI API key (input hidden): ").strip()
    if entered_key:
        os.environ["OPENAI_API_KEY"] = entered_key

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set for this Jupyter kernel.")


In [15]:
gpt_celltype_adata = data.copy()

#normalize and high variable genes (HVGs) calculated
gpt_celltype_adata=ov.pp.preprocess(gpt_celltype_adata,mode='shiftlog|pearson',n_HVGs=2000,)

#save the whole genes and filter the non-HVGs
gpt_celltype_adata.raw = gpt_celltype_adata
gpt_celltype_adata = gpt_celltype_adata[:, gpt_celltype_adata.var.highly_variable_features]

#scale the adata.X
ov.pp.scale(gpt_celltype_adata)

#Dimensionality Reduction
ov.pp.pca(gpt_celltype_adata,layer='scaled',n_pcs=50)

#Neighbourhood graph construction
sc.pp.neighbors(gpt_celltype_adata, n_neighbors=15, n_pcs=50,
               use_rep='scaled|original|X_pca')

#clusters
sc.tl.leiden(gpt_celltype_adata)

#find marker
sc.tl.dendrogram(gpt_celltype_adata,'leiden',use_rep='scaled|original|X_pca')
sc.tl.rank_genes_groups(gpt_celltype_adata, 'leiden', use_rep='scaled|original|X_pca',
                        method='wilcoxon',use_raw=False,)

🔍 [2026-04-07 21:56:00] Running preprocessing in 'cpu' mode...
Begin robust gene identification
    After filtration, 238/238 genes are kept.
    Among 238 genes, 238 genes are robust.
✅ Robust gene identification completed successfully.
Begin size normalization: shiftlog and HVGs selection pearson

🔍 Count Normalization:
   Target sum: 500000.0
   Exclude highly expressed: True
   Max fraction threshold: 0.2
   ⚠️ Excluding 1 highly-expressed genes from normalization computation
   Excluded genes: ['MYH6']

✅ Count Normalization Completed Successfully!
   ✓ Processed: 228,635 cells × 238 genes
   ✓ Runtime: 0.34s

🔍 Highly Variable Genes Selection (Experimental):
   Method: pearson_residuals
   Target genes: 2,000
   Theta (overdispersion): 100

✅ Experimental HVG Selection Completed Successfully!
   ✓ Selected: 238 highly variable genes out of 238 total (100.0%)
   ✓ Results added to AnnData object:
     • 'highly_variable': Boolean vector (adata.var)
     • 'highly_variable_rank': F

... storing 'celltypist_pred_harmonized' as categorical


In [16]:
all_markers=ov.single.get_celltype_marker(gpt_celltype_adata,clustertype='leiden',rank=True,
                                          key='rank_genes_groups',
                                          foldchange=2,)

result = ov.single.gptcelltype(all_markers, tissuename='heart', speciename='human',model='gpt-5.1', provider='openai')

...get cell type marker
Note: AGI API key found: returning the cell type annotations.


HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://api.openai.co

Note: It is always recommended to check the results returned by GPT-4 in case of AI hallucination, before going to downstream analysis.


In [59]:
gptcelltype_harmonized_preds = gpt_celltype_adata.obs.leiden.map(lambda x : mapping['mapping_gptcelltype'][result[x]])

data.obs['gptcelltype_pred_harmonized'] = gptcelltype_harmonized_preds

### Extract TissueAgent Predictions

In [27]:
agents_adata = sc.read_h5ad('../data/data/annotated_object.h5ad') 

data.obs['tissueagent_pred_harmonized'] =  agents_adata.obs.harmony_predicted_cell_type.map(lambda x: mapping['mapping_tissueagent'][x])

In [33]:
data.write_h5ad('../data/dataset/overall_merfish_w_all_predictions.h5ad')

### Compute Evaluation Metrics w/ Predictions

In [34]:
from sklearn.metrics import precision_score

data = sc.read_h5ad('../data/dataset/overall_merfish_w_all_predictions.h5ad')

In [35]:
methods = ['tissueagent_pred_harmonized', 'gptcelltype_pred_harmonized',
       'celltypist_pred_harmonized']

In [36]:
eval_metrics = {'accuracy':{},'macro_precision':{}}
gt = data.obs['populations'].map(lambda x: mapping['mapping_ground_truth'][x])

for m in methods:
    preds = data.obs[m].astype('str')
    eval_metrics['accuracy'][m] = round(float((gt == preds).mean() * 100),2)
    eval_metrics['macro_precision'][m] = round(precision_score(gt.to_list(),preds.to_list(),average='macro', zero_division=0)* 100,2)


eval_metrics

{'accuracy': {'tissueagent_pred_harmonized': 77.5,
  'gptcelltype_pred_harmonized': 81.97,
  'celltypist_pred_harmonized': 42.98},
 'macro_precision': {'tissueagent_pred_harmonized': 44.87,
  'gptcelltype_pred_harmonized': 36.93,
  'celltypist_pred_harmonized': 32.08}}